In [1]:
import pandas as pd
import os
import json
import numpy as np

In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.pyplot import cm
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

## Get MVP significant SNPs

In [3]:
sumstats_dir="../../../discovery/data/summarystats/full/"

In [4]:
eur=pd.read_csv(sumstats_dir+"compiled.eur.release4.testosterone.glm.linear",delimiter="\t")
afr=pd.read_csv(sumstats_dir+"compiled.afr.release4.testosterone.glm.linear",delimiter="\t")
his=pd.read_csv(sumstats_dir+"compiled.his.release4.testosterone.glm.linear",delimiter="\t")

/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3049: DtypeWarning: Columns (0,4) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [5]:
eur_sig=eur[eur["P"]<.00000005]
afr_sig=afr[afr["P"]<.00000005]
his_sig=his[his["P"]<.00000005]

In [6]:
eur_lst=eur_sig["ID"].tolist()+[x.rsplit(":",2)[0]+":"+x.split(":")[3]+":"+x.split(":")[2] for x in eur_sig["ID"].tolist()]
afr_lst=afr_sig["ID"].tolist()+[x.rsplit(":",2)[0]+":"+x.split(":")[3]+":"+x.split(":")[2] for x in afr_sig["ID"].tolist()]
his_lst=his_sig["ID"].tolist()+[x.rsplit(":",2)[0]+":"+x.split(":")[3]+":"+x.split(":")[2] for x in his_sig["ID"].tolist()]
snp_lst=eur_lst+afr_lst+his_lst

In [8]:
compiled_cis=pd.DataFrame()
file=[]
num_eqtls=[]
num_total_eqtls=[]

gtex_dir="/cellar/users/mpagadal/data/gtex/"
gtex_eqtl="/cellar/users/mpagadal/data/gtex/GTEx_Analysis_v8_eQTL_EUR/"

files=[x for x in os.listdir(gtex_eqtl) if "pairs" in x]

mapping=pd.read_csv(gtex_dir+"GTEx_Analysis_2017-06-05_v8_WholeGenomeSeq_838Indiv_Analysis_Freeze.lookup_table.txt",delimiter="\t")
mp_b38_b37=dict(zip(mapping.variant_id,mapping.variant_id_b37))

for x in files:
    print(x)
    eqtl=pd.read_csv(gtex_eqtl+x,delimiter="\t")
    eqtl["variant_b37"] = eqtl["variant_id"].map(mp_b38_b37)
    eqtl["variant"] = eqtl["variant_b37"].str.split("_").str[0]+":"+eqtl["variant_b37"].str.split("_").str[1]+":"+eqtl["variant_b37"].str.split("_").str[2]+":"+eqtl["variant_b37"].str.split("_").str[3]
    eqtl["cell"]=x.split(".v8")[0]
    
    eqtl_filt=eqtl[eqtl["variant"].isin(snp_lst)]
    
    file.append(x)
    num_eqtls.append(len(eqtl_filt))
    num_total_eqtls.append(len(eqtl))
    
    compiled_cis=compiled_cis.append(eqtl_filt)

Uterus.v8.EUR.signif_pairs.txt
Colon_Sigmoid.v8.EUR.signif_pairs.txt
Brain_Frontal_Cortex_BA9.v8.EUR.signif_pairs.txt
Cells_Cultured_fibroblasts.v8.EUR.signif_pairs.txt
Brain_Cerebellum.v8.EUR.signif_pairs.txt
Colon_Transverse.v8.EUR.signif_pairs.txt
Brain_Spinal_cord_cervical_c-1.v8.EUR.signif_pairs.txt
Adrenal_Gland.v8.EUR.signif_pairs.txt
Cells_EBV-transformed_lymphocytes.v8.EUR.signif_pairs.txt
Esophagus_Mucosa.v8.EUR.signif_pairs.txt
Artery_Tibial.v8.EUR.signif_pairs.txt
Adipose_Visceral_Omentum.v8.EUR.signif_pairs.txt
Small_Intestine_Terminal_Ileum.v8.EUR.signif_pairs.txt
Muscle_Skeletal.v8.EUR.signif_pairs.txt
Esophagus_Gastroesophageal_Junction.v8.EUR.signif_pairs.txt
Thyroid.v8.EUR.signif_pairs.txt
Heart_Left_Ventricle.v8.EUR.signif_pairs.txt
Brain_Anterior_cingulate_cortex_BA24.v8.EUR.signif_pairs.txt
Esophagus_Muscularis.v8.EUR.signif_pairs.txt
Artery_Aorta.v8.EUR.signif_pairs.txt
Breast_Mammary_Tissue.v8.EUR.signif_pairs.txt
Kidney_Cortex.v8.EUR.signif_pairs.txt
Heart_Atria

In [9]:
gtex_counts=pd.DataFrame({"file":file,"total":num_total_eqtls,"significant":num_eqtls})
gtex_counts["fraction"]=gtex_counts["significant"]/gtex_counts["total"]
gtex_counts.sort_values(by="fraction")

,file,total,significant,fraction
32,Brain_Substantia_nigra.v8.EUR.signif_pairs.txt,241171,9,0.000037
39,Liver.v8.EUR.signif_pairs.txt,561530,296,0.000527
21,Kidney_Cortex.v8.EUR.signif_pairs.txt,100336,54,0.000538
6,Brain_Spinal_cord_cervical_c-1.v8.EUR.signif_p...,394911,249,0.000631
16,Heart_Left_Ventricle.v8.EUR.signif_pairs.txt,1262657,856,0.000678
12,Small_Intestine_Terminal_Ileum.v8.EUR.signif_p...,513930,408,0.000794
2,Brain_Frontal_Cortex_BA9.v8.EUR.signif_pairs.txt,718330,575,0.000800
36,Brain_Putamen_basal_ganglia.v8.EUR.signif_pair...,645689,536,0.000830
9,Esophagus_Mucosa.v8.EUR.signif_pairs.txt,2337514,1999,0.000855
4,Brain_Cerebellum.v8.EUR.signif_pairs.txt,1428248,1243,0.000870


### Extract significant genes

In [10]:
compiled_cis["gene_id"]=compiled_cis["phenotype_id"].str.split(".").str[0]

with open('/cellar/users/mpagadal/resources/tcga/ensembl_map.json', 'r') as f:
    ensembl_dict = json.load(f)
ensembl_dict={k.split(".")[0]:v for k,v in ensembl_dict.items()}

compiled_cis["gtex gene"]=compiled_cis["gene_id"].map(ensembl_dict)

In [11]:
len(compiled_cis["gtex gene"].unique())

240

In [12]:
genes=pd.DataFrame({"gene":compiled_cis["phenotype_id"].unique().tolist()})
genes.to_csv("gtex.significant.genes.txt",header=None,index=None,sep="\t")